# TripMe Sri Lanka — Travel-Time / Distance Collector

**Goal:** fill a real gap in `data/processed/places.json` for the Trip Planner
feature - `lat`/`lng` in the source dataset are only reliable at the
district level (1283 unique coordinate pairs across 2278 places), so there's
no real travel-time/distance signal between places today. This notebook
fills that in using **OSRM** (Open Source Routing Machine, free, no API key,
no usage cost - unlike Google Maps Distance Matrix).

**Scope:** same-district place pairs only (~158k pairs across 25 districts),
not the full 2278x2278 cross-district matrix (2.5M+ pairs) - that's the
practical scope, since Trip Planner routes are built stop-by-stop within a
district (see `02_generate_instructions.py`'s day-plan builders), and the
existing province-grouping already handles multi-district routing without
needing exact cross-district driving distances.

**How it queries efficiently:** OSRM's `table` service computes an NxN
travel-time/distance matrix in one HTTP request - **but the free public
demo server caps requests at ~100 coordinates** (confirmed by testing: 100
succeeds, 110 returns `TooBig`), well under several of our districts (Kandy
has 289 places, Kurunagala 217, Hambanthota 164, Kalutara 157, Galle 153,
Badulla 144). Districts over the limit are automatically split into ~45-place
chunks, queried pairwise via OSRM's `sources`/`destinations` parameters
(each chunk-pair request only sends the coordinates for that pair of chunks,
staying under the 100 limit) and reassembled into the full matrix - e.g.
Kandy's 289 places become 7 chunks -> 49 chunk-pair requests, confirmed
working end to end (83,232/83,232 expected pairs collected in testing).

**Output:** `data/processed/travel_times.json`, keyed by `district_id`, each
holding a place-id-indexed distance/duration matrix - a lookup structure the
Trip Planner scenario builders can use directly (real minutes-between-stops
instead of no distance signal at all).

**Runtime:** no GPU needed, pure HTTP calls to the free public OSRM demo
server. Small districts are one request each; the 6 large districts needing
chunking add up to roughly 150 additional requests total. A few minutes
overall, resumable per chunk-pair if interrupted.


## 1. Setup

In [ ]:
!pip install -q -U requests tqdm
print("Dependencies installed.")

In [ ]:
import json
import time
from pathlib import Path

import requests
from tqdm.auto import tqdm

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_FILE = OUTPUT_DIR / "travel_times.json"
PROGRESS_FILE = OUTPUT_DIR / "travel_times_progress.json"

# Path to places.json - upload it as a Kaggle Dataset and adjust this path,
# same pattern as collect_osm_places.ipynb's EXISTING_PLACES_PATH.
PLACES_PATH = Path("/kaggle/input/tripme-places/places.json")
if not PLACES_PATH.exists():
    PLACES_PATH = Path("data/processed/places.json")  # local run fallback

# OSRM's free public demo server - no key required. If you hit rate limits
# or want production reliability, self-host OSRM or point this at a paid
# provider's table endpoint instead (interface is different, but the rest
# of this notebook's structure would carry over).
OSRM_TABLE_URL = "https://router.project-osrm.org/table/v1/driving"
REQUEST_DELAY = 1.5  # seconds between district requests - be polite to the free public instance
REQUEST_TIMEOUT = 120  # larger districts (Kandy: 289 points) can take a while server-side

HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update({
    "User-Agent": "TripMeSriLankaTravelTimeCollector/1.0 (Kaggle notebook; contact: n/a)",
})

print("Places path:", PLACES_PATH)
print("Output will be written to:", OUTPUT_FILE)

## 2. Load places, group by district

In [ ]:
with open(PLACES_PATH, encoding="utf-8") as f:
    places = json.load(f)

places_by_district = {}
for p in places:
    places_by_district.setdefault(p["district_id"], []).append(p)

print(f"{len(places)} places across {len(places_by_district)} districts.")
for d, ps in sorted(places_by_district.items(), key=lambda x: -len(x[1])):
    n = len(ps)
    print(f"  {d:15s} {n:4d} places -> {n*(n-1)//2:6d} pairs")

## 3. OSRM table request, chunked

OSRM's table service takes a semicolon-separated list of `lng,lat` coordinates
(note the order - longitude first, unlike most of this project's `lat, lng`
convention). For a chunk pair (source chunk, destination chunk), only that
pair's combined coordinates are sent in the URL, with `sources`/`destinations`
index parameters pointing at which part of that combined list is which -
this keeps every single request under OSRM's ~100-coordinate cap regardless
of the district's total size.


In [ ]:
CHUNK_SIZE = 45  # two chunks combined = 90 coordinates, safely under the ~100 limit


def chunked(lst, size):
    return [lst[i:i + size] for i in range(0, len(lst), size)]


def osrm_table_chunk(src_places, dst_places, retries=3):
    """One chunk-pair request: distances/durations from every place in
    src_places to every place in dst_places. src_places and dst_places may
    be the same list (diagonal chunk pair) or different (off-diagonal)."""
    combined = src_places + dst_places
    coords = ";".join(f"{p['lng']},{p['lat']}" for p in combined)
    sources = ";".join(str(i) for i in range(len(src_places)))
    destinations = ";".join(str(i) for i in range(len(src_places), len(combined)))
    url = f"{OSRM_TABLE_URL}/{coords}"
    params = {"annotations": "duration,distance", "sources": sources, "destinations": destinations}

    for attempt in range(retries):
        try:
            r = HTTP_SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200:
                try:
                    data = r.json()
                except ValueError:
                    print(f"    non-JSON 200 response (attempt {attempt+1}), retrying...")
                    time.sleep(10 * (attempt + 1))
                    continue
                if data.get("code") != "Ok":
                    print(f"    OSRM returned code={data.get('code')} (attempt {attempt+1}): "
                          f"{data.get('message', '')}")
                    time.sleep(5 * (attempt + 1))
                    continue
                return data
            if r.status_code == 429 or r.status_code == 504:
                time.sleep(10 * (attempt + 1))
                continue
            r.raise_for_status()
        except requests.RequestException as e:
            print(f"    request failed (attempt {attempt+1}): {e}")
            time.sleep(5 * (attempt + 1))
    print("    giving up on this chunk pair after retries.")
    return None


print("OSRM chunked table helper ready.")


## 4. Resumable progress state

Tracked per **(district, chunk_i, chunk_j)** - the actual unit of work now
that large districts are split into multiple chunk-pair requests - so a
disconnect partway through a large district's ~49 chunk-pair requests only
re-does the missing ones, not the whole district.


In [ ]:
def load_progress():
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    return {"completed_chunk_pairs": [], "completed_districts": []}


def save_progress(progress):
    PROGRESS_FILE.write_text(json.dumps(progress, indent=2), encoding="utf-8")


def load_results():
    if OUTPUT_FILE.exists():
        return json.loads(OUTPUT_FILE.read_text(encoding="utf-8"))
    return {}


def chunk_pair_key(district, i, j):
    return f"{district}::{i}::{j}"


progress = load_progress()
results = load_results()
print(f"Resuming: {len(progress['completed_districts'])} districts fully done, "
      f"{len(progress['completed_chunk_pairs'])} individual chunk-pairs done.")


## 5. Main collection loop

For each district, splits its places into chunks of `CHUNK_SIZE`, then makes
one OSRM request per (chunk_i, chunk_j) pair covering that district's full
NxN matrix (chunk_i == chunk_j pairs cover the diagonal block; i != j pairs
cover the rest). Each response is converted into a `place_id -> place_id ->
{duration_min, distance_km}` lookup and merged into `results[district]` -
keyed by real place IDs, not array indices, so the output is self-contained
and doesn't depend on places.json's list order to be usable later.


In [ ]:
unmatched_note = (
    "Some OSRM responses report 'null' for a pair with no road-network match "
    "nearby (e.g. a point that snapped to an isolated trail) - those pairs "
    "are simply omitted from the output rather than filled with a fabricated "
    "number."
)
print(unmatched_note)

# Pre-count total chunk-pairs across all districts so the overall bar has a real total.
_district_chunks = {d: chunked(ps, CHUNK_SIZE) for d, ps in places_by_district.items() if len(ps) >= 2}
_total_chunk_pairs = sum(len(chs) * len(chs) for chs in _district_chunks.values())
_already_done = len(progress["completed_chunk_pairs"])

overall_bar = tqdm(total=_total_chunk_pairs, initial=_already_done, desc="Chunk-pairs", unit="pair")
for district, chunks in _district_chunks.items():
    if district in progress["completed_districts"]:
        overall_bar.update(len(chunks) * len(chunks) -
                            sum(1 for i in range(len(chunks)) for j in range(len(chunks))
                                if chunk_pair_key(district, i, j) in progress["completed_chunk_pairs"]))
        continue

    district_matrix = results.get(district, {})
    district_failed = False

    for i, src_chunk in enumerate(chunks):
        for j, dst_chunk in enumerate(chunks):
            key = chunk_pair_key(district, i, j)
            if key in progress["completed_chunk_pairs"]:
                continue

            data = osrm_table_chunk(src_chunk, dst_chunk)
            if data is None:
                district_failed = True  # this district won't be marked fully complete
                overall_bar.update(1)
                continue

            durations = data.get("durations") or []
            distances = data.get("distances") or []
            for a, from_p in enumerate(src_chunk):
                row = district_matrix.setdefault(from_p["id"], {})
                for b, to_p in enumerate(dst_chunk):
                    if from_p["id"] == to_p["id"]:
                        continue
                    dur = durations[a][b] if a < len(durations) and b < len(durations[a]) else None
                    dist = distances[a][b] if a < len(distances) and b < len(distances[a]) else None
                    if dur is None or dist is None:
                        continue
                    row[to_p["id"]] = {
                        "duration_min": round(dur / 60, 1),
                        "distance_km": round(dist / 1000, 2),
                    }

            progress["completed_chunk_pairs"].append(key)
            results[district] = district_matrix
            OUTPUT_FILE.write_text(json.dumps(results, ensure_ascii=False), encoding="utf-8")
            save_progress(progress)
            overall_bar.update(1)
            overall_bar.set_postfix_str(f"{district} ({i+1},{j+1})/{len(chunks)}")
            time.sleep(REQUEST_DELAY)

    if not district_failed:
        progress["completed_districts"].append(district)
        save_progress(progress)

overall_bar.close()
print(f"\nCompleted {len(progress['completed_districts'])} / {len(_district_chunks)} districts fully.")


## 6. Sanity check + summary

In [ ]:
total_pairs = sum(len(row) for district_matrix in results.values() for row in district_matrix.values())
print(f"Districts with data: {len(results)}")
print(f"Total directed pairs stored: {total_pairs}")

if "Kandy" in results:
    kandy_ids = list(results["Kandy"].keys())[:2]
    if len(kandy_ids) == 2:
        a, b = kandy_ids
        pair = results["Kandy"][a].get(b)
        print(f"\nSample (Kandy, {a} -> {b}): {pair}")

print(f"\nSaved -> {OUTPUT_FILE}")
print("Next steps:")
print("  1. Download travel_times.json")
print("  2. Drop it into data/processed/ in the project")
print("  3. Use it in the Trip Planner scenario builders (02_generate_instructions.py) "
      "to pick genuinely nearby stops for the same day instead of a random shuffle, "
      "and/or add a 'travel_time_between_stops' field to the structured output.")
print()
print("Attribution reminder: routing computed via OSRM (https://project-osrm.org), "
      "road network data (c) OpenStreetMap contributors (ODbL).")